# RF-regularizer diagnostics: single-blob for `Factorized2d`

Runs the three diagnostics from `rf_regularizers_handoff.md` ("Diagnostics to run
first") against a trained checkpoint, to decide whether the **peak-distance
penalty** or the **concavity penalty** is better targeted before spending time
tuning either one.

1. **Ramp-R² of `W`** — is the per-pixel ramp image effectively linear (curvature-free),
   so second blobs must come from `b` alone?
2. **Width explained by peak position** — is mask width (`r50`) a function of the
   peak pixel only, or is there genuine per-neuron width variation?
3. **Cross-check** — do large-`|source_grid|` neurons carry the blob/width effects,
   and which animal has the widest cortex-coordinate spread?

Run **per session** (= per animal here, all 11 sessions are distinct animals) —
the handoff explicitly warns against pooling cortex coordinates across animals
with `np.vstack`.

Checkpoint: `11mice_seed37.pth` (repo root, sibling of `sensorium/` and
`neuralpredictors/`). Unlike the earlier `fourier_*` checkpoints, this one's
readout matches the current `Factorized2d` class exactly (`spatial_w`/`spatial_b`/
`source_grid` as direct parameters, no basis/mlp reparametrization), so the
diagnostics below use the real class's `.spatial` property directly instead of
reconstructing it by hand.

In [ ]:
import re
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy import ndimage

from neuralpredictors.layers.readouts import Factorized2d

torch.set_grad_enabled(False)


## Load checkpoint and get `b`, `Wx`, `Wy`, mask per session

This checkpoint's readout matches `neuralpredictors/layers/readouts/factorized.py`'s
`Factorized2d` directly — each `readout.<session>` stores `spatial_w` `(2, h, w)`,
`spatial_b` `(1, h, w)`, and `source_grid` `(n_neurons, 2)` as plain parameters/buffer
(no basis/mlp reparametrization like the earlier `fourier_*` checkpoints). So instead
of reconstructing `rf`/mask by hand, we instantiate the real `Factorized2d` per
session, copy in its `spatial_w`/`spatial_b`/`source_grid`, and read `.spatial`
straight off the class — guaranteed to match production's smoothing + softmax
exactly, not a re-implementation of it.

**Assumption flagged:** the checkpoint has no `kernel_sigma`/`log_temp` keys, so we
can't recover the exact smoothing kernel / softmax temperature the run actually used
(those are only saved when `smoothness_reg_weight > 0` / `temp_per_neuron=True`) —
they're plain non-learnable floats otherwise and never hit the state dict either way.
We use the class defaults (`kernel_size=7`, `kernel_sigma=2.0`, `temperature=1.0`).
This doesn't affect diagnostic 1 (computed directly on `spatial_w`/`spatial_b`,
pre-smoothing/softmax); it could shift absolute `r50` values in diagnostic 2, but not
the position-explained-variance ratio.

In [ ]:
CKPT_PATH = "../../11mice_seed37.pth"

sd = torch.load(CKPT_PATH, map_location="cpu")
session_keys = sorted({
    m.group(1) for k in sd if (m := re.match(r"readout\.([^.]+)\.spatial_w$", k))
})
print(f"{len(session_keys)} sessions:", session_keys)


In [ ]:
def reconstruct_session(sd, session):
    prefix = f"readout.{session}."
    spatial_w = sd[prefix + "spatial_w"]   # (2, h, w)
    spatial_b = sd[prefix + "spatial_b"]   # (1, h, w)
    source_grid = sd[prefix + "source_grid"]   # (n, 2), already centered/max-abs-normalized
    n = sd[prefix + "bias"].shape[0]
    c = sd[prefix + "_features"].shape[1]
    h, w = spatial_w.shape[1:]

    readout = Factorized2d(
        in_shape=(c, h, w), outdims=n, bias=True,
        source_grid=source_grid.numpy(), feature_reg_weight=1.0,
    )
    readout.spatial_w.data.copy_(spatial_w)
    readout.spatial_b.data.copy_(spatial_b)

    mask = readout.spatial.detach()

    return {
        "n": n,
        "h": h,
        "w": w,
        "source_grid": source_grid,
        "b": spatial_b[0],
        "Wx": spatial_w[0],
        "Wy": spatial_w[1],
        "mask": mask,
    }


sessions_data = {s: reconstruct_session(sd, s) for s in session_keys}

summary = pd.DataFrame([
    {
        "session": s,
        "n_neurons": d["n"],
        "h": d["h"],
        "w": d["w"],
        "max_abs_coord": d["source_grid"].norm(dim=1).max().item(),
    }
    for s, d in sessions_data.items()
])
summary


## Diagnostic 1: Ramp-R² of `W`

Least-squares fit each `Wx`/`Wy` image against the pixel-coordinate basis
`[p_x, p_y, 1]` and report `1 - resid.var() / W.var()`. Near 1 -> `W` is
effectively a linear ramp (no curvature of its own) -> any curvature in `rf_n` comes
from `b`, and the concavity penalty only has to fix `b` (cheap, well-targeted).
Well below 1 -> `W` carries real curvature that contributes per-neuron blob width /
nonlinear retinotopy -> the concavity penalty would charge that legitimate
curvature too; prefer the peak-distance penalty.

In [ ]:
def ramp_r2(img):
    h, w = img.shape
    py, px = torch.meshgrid(
        torch.arange(h, dtype=img.dtype), torch.arange(w, dtype=img.dtype), indexing="ij"
    )
    A = torch.stack([px.reshape(-1), py.reshape(-1), torch.ones(h * w, dtype=img.dtype)], dim=1)
    y = img.reshape(-1, 1)
    coef = torch.linalg.lstsq(A, y).solution
    resid = y - A @ coef
    return (1 - resid.var(unbiased=False) / y.var(unbiased=False)).item()


ramp_rows = []
for s, d in sessions_data.items():
    r2x, r2y = ramp_r2(d["Wx"]), ramp_r2(d["Wy"])
    ramp_rows.append({"session": s, "ramp_r2_Wx": r2x, "ramp_r2_Wy": r2y, "ramp_r2_mean": (r2x + r2y) / 2})

ramp_df = pd.DataFrame(ramp_rows).sort_values("ramp_r2_mean")
ramp_df


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(ramp_df))
ax.bar(x - 0.2, ramp_df["ramp_r2_Wx"], width=0.4, label="Wx")
ax.bar(x + 0.2, ramp_df["ramp_r2_Wy"], width=0.4, label="Wy")
ax.axhline(0.9, color="k", linestyle="--", linewidth=1, label="R²=0.9")
ax.set_xticks(x)
ax.set_xticklabels(ramp_df["session"], rotation=45, ha="right")
ax.set_ylabel("ramp R²")
ax.set_title("Diagnostic 1: is W a clean linear ramp?")
ax.legend()
plt.tight_layout()
plt.show()


## Diagnostic 2: width explained by peak position

For each neuron, find its peak pixel and its half-mass radius `r50` (the radius
from the peak that first accumulates 50% of the mask's total mass). Group neurons
by exact peak pixel (bins with >= 5 neurons) and report
`1 - within_bin_var / total_var` on `r50`. Near 1 -> width is a pure function of
retinotopic position (no leftover per-neuron variation to explain) -> favors the
peak-distance penalty (leaves the "one blob per position" family untouched). Low ->
real per-neuron width variation exists beyond position -> constraining width would
cost real capacity.

In [ ]:
def compute_peak_and_r50(mask, chunk=2000):
    n, h, w = mask.shape
    yy, xx = torch.meshgrid(
        torch.arange(h, dtype=mask.dtype), torch.arange(w, dtype=mask.dtype), indexing="ij"
    )
    peak_y = torch.empty(n, dtype=torch.long)
    peak_x = torch.empty(n, dtype=torch.long)
    r50 = torch.empty(n, dtype=mask.dtype)

    for s in range(0, n, chunk):
        mc = mask[s : s + chunk]
        b = mc.shape[0]
        idx = mc.reshape(b, -1).argmax(1)
        py, px = idx // w, idx % w
        peak_y[s : s + b], peak_x[s : s + b] = py, px

        dy2 = (yy[None] - py[:, None, None].to(mask.dtype)) ** 2
        dx2 = (xx[None] - px[:, None, None].to(mask.dtype)) ** 2
        d = (dy2 + dx2).sqrt().reshape(b, -1)
        mflat = mc.reshape(b, -1)

        order = d.argsort(dim=1)
        d_sorted = torch.gather(d, 1, order)
        m_sorted = torch.gather(mflat, 1, order)
        cum = m_sorted.cumsum(dim=1)
        first_idx = (cum >= 0.5).float().argmax(dim=1)
        r50[s : s + b] = torch.gather(d_sorted, 1, first_idx[:, None]).squeeze(1)

    return peak_y, peak_x, r50


def width_explained_by_position(peak_y, peak_x, r50, min_bin=5):
    groups = defaultdict(list)
    for i, key in enumerate(zip(peak_y.tolist(), peak_x.tolist())):
        groups[key].append(i)

    kept_idx, within, n_total = [], 0.0, 0
    for idxs in groups.values():
        if len(idxs) < min_bin:
            continue
        vals = r50[torch.tensor(idxs)]
        within += vals.var(unbiased=False).item() * len(idxs)
        n_total += len(idxs)
        kept_idx.extend(idxs)

    if n_total < 2:
        return float("nan"), 0
    total_var = r50[torch.tensor(kept_idx)].var(unbiased=False).item()
    within_var = within / n_total
    return 1 - within_var / total_var, n_total


width_rows = []
for s, d in sessions_data.items():
    peak_y, peak_x, r50 = compute_peak_and_r50(d["mask"])
    d["peak_y"], d["peak_x"], d["r50"] = peak_y, peak_x, r50
    r2, n_kept = width_explained_by_position(peak_y, peak_x, r50)
    width_rows.append({
        "session": s,
        "width_r2": r2,
        "n_neurons_in_bins>=5": n_kept,
        "frac_neurons_in_bins>=5": n_kept / d["n"],
    })

width_df = pd.DataFrame(width_rows).sort_values("width_r2")
width_df


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(width_df))
ax.bar(x, width_df["width_r2"])
ax.axhline(0.9, color="k", linestyle="--", linewidth=1, label="R²=0.9")
ax.set_xticks(x)
ax.set_xticklabels(width_df["session"], rotation=45, ha="right")
ax.set_ylabel("width-explained-by-position R²")
ax.set_title("Diagnostic 2: is r50 a function of peak position only?")
ax.legend()
plt.tight_layout()
plt.show()


## Diagnostic 3: cross-checks

- Do large-`|source_grid|` neurons carry the second-blob / width effects?
- Which animal (session) has the widest cortex-coordinate spread — cross-check
  against the handoff's claim that animal 3 / peripheral neurons show the
  strongest second-blob artifact.

Blob count uses connected components on the mask thresholded at a low fraction of
its own peak (this is *only* a diagnostic proxy for "how many separate blobs does
this neuron's mask have", not the peak-distance penalty itself).

In [ ]:
def blob_counts(mask, rel_thresh=0.2):
    mask_np = mask.numpy()
    counts = np.empty(mask_np.shape[0], dtype=int)
    for i in range(mask_np.shape[0]):
        thr = mask_np[i].max() * rel_thresh
        _, num = ndimage.label(mask_np[i] > thr)
        counts[i] = num
    return counts


def safe_corr(a, b):
    # nan (not a divide-by-zero warning) when a column is constant, e.g. zero neurons
    # flagged as multiblob at this threshold — a real outcome for well-smoothed masks.
    if a.std() == 0 or b.std() == 0:
        return float("nan")
    return np.corrcoef(a, b)[0, 1]


crosscheck_rows = []
for s, d in sessions_data.items():
    counts = blob_counts(d["mask"])
    coord_norm = d["source_grid"].norm(dim=1).numpy()
    r50 = d["r50"].numpy()

    crosscheck_rows.append({
        "session": s,
        "frac_multiblob": (counts > 1).mean(),
        "corr(|coord|, multiblob)": safe_corr(coord_norm, (counts > 1).astype(float)),
        "corr(|coord|, r50)": safe_corr(coord_norm, r50),
        "mean_abs_coord": d["source_grid"].abs().mean().item(),
        "max_abs_coord": coord_norm.max(),
    })

crosscheck_df = pd.DataFrame(crosscheck_rows).sort_values("max_abs_coord", ascending=False)
crosscheck_df


In [ ]:
widest = crosscheck_df.iloc[0]["session"]
print(f"Widest cortex-coordinate spread: session {widest}")
print("(cross-check this against whichever session the prior 'animal 3' report referred to —")
print(" session->animal numbering here follows os.listdir() order in train.py, which is not")
print(" a fixed/deterministic canonical index, so it isn't re-derived here.)")


## Summary and recommendation

Combine diagnostics 1 and 2 per session. Decision rule taken directly from the
handoff — **diagnostic 1 (`ramp_r2_mean`) is the decisive one**:

- `ramp_r2_mean` near 1 -> curvature lives in `b` alone -> **concavity penalty**
  is cheap and well-targeted.
- `ramp_r2_mean` well below 1 -> `W` carries real curvature that is doing work
  (per-neuron blob width / nonlinear retinotopy) -> concavity penalty would charge
  that legitimate curvature too -> **peak-distance penalty** is the better-targeted
  option.

`width_r2` (diagnostic 2) doesn't flip that call — it only refines *how* to apply
the peak-distance penalty: near 1 means width is ~purely a function of peak
position, so a single global `radius` is safe; low means genuine per-neuron width
variation exists, so a fixed global radius risks charging legitimately wide (but
still single-blob) neurons, and `radius` should be set conservatively (e.g. a
per-animal quantile of observed widths) rather than one constant.

This is one seed only (`seed37`) — a single-checkpoint read, not the weight-sweep
+ cross-seed-consistency protocol the handoff asks for before committing to a
weight. Treat the recommendation column as a per-session prior to sweep around,
not a verdict.

In [ ]:
def recommend(row, ramp_thresh=0.7, width_thresh=0.7):
    if row["ramp_r2_mean"] >= ramp_thresh:
        return "penalty 2 (concavity) — W is close to a clean ramp"
    elif row["width_r2"] >= width_thresh:
        return "penalty 1 (peak-distance), single global radius OK — width is ~purely positional"
    else:
        return "penalty 1 (peak-distance) — W has real curvature; width also varies genuinely per-neuron, so set radius conservatively (e.g. per-animal quantile) rather than one fixed constant"


final_df = (
    ramp_df.merge(width_df, on="session")
    .merge(crosscheck_df[["session", "frac_multiblob", "max_abs_coord"]], on="session")
    .sort_values("max_abs_coord", ascending=False)
    .reset_index(drop=True)
)
final_df["recommendation"] = final_df.apply(recommend, axis=1)
final_df


In [ ]:
print(f"ramp_r2_mean across sessions: {final_df['ramp_r2_mean'].min():.3f} - {final_df['ramp_r2_mean'].max():.3f}")
print(f"width_r2 across sessions:     {final_df['width_r2'].min():.3f} - {final_df['width_r2'].max():.3f}")
print(final_df["recommendation"].value_counts())


## Caveats

- Single checkpoint / single seed — no cross-seed consistency (ARI / kNN / AA
  alpha-column stability) computed here; that's the next step once a penalty is
  chosen, per the handoff's evaluation protocol.
- `kernel_size=7`, `kernel_sigma=2.0`, `T=1` are `Factorized2d` class defaults, not
  necessarily what this run actually used (see load-time note above — they're
  non-learnable and not saved in the checkpoint either way). Affects absolute `r50`
  scale in diagnostic 2, not its position-explained-variance ratio; doesn't affect
  diagnostic 1 at all (computed pre-smoothing/softmax, directly on `spatial_w`/`spatial_b`).
- `blob_counts` threshold (`0.2 * peak`) is a coarse proxy for illustration in
  diagnostic 3, not a calibrated blob-detector; don't reuse its exact threshold
  elsewhere without checking sensitivity.
- Everything above is computed per-session/per-animal by construction (each
  session here is a distinct animal) — consistent with the handoff's warning
  against pooling cortex coordinates across animals.